# Badge 3: Data Application Builders Workshop

## ❓ What is Streamlit in Snowflake?


In [ ]:
%%sql -r dataframe_1
use role sysadmin;
use warehouse COMPUTE_WH;
create database smoothies;


In [ ]:
%%sql -r dataframe_2
use role sysadmin;
use database smoothies;
use warehouse COMPUTE_WH;

create table fruit_options (
    fruit_id bigint primary key,
    fruit_name varchar(50) 
)

In [ ]:
%%sql -r dataframe_3
create file format smoothies.public.two_headerrow_pct_delim
  type = CSV,
  skip_header = 2,
  field_delimiter = '%',
  trim_space = TRUE
;

In [ ]:
%%sql -r dataframe_4
SELECT $1, $2, $3, $4, $5
FROM @smoothies.public.my_uploaded_files/fruits_available_for_smoothies.txt
(FILE_FORMAT => smoothies.public.two_headerrow_pct_delim );

In [ ]:
%%sql -r dataframe_5
COPY INTO smoothies.public.fruit_options
FROM (select $2 as FRUIT_ID, $1 as FRUIT_NAME 
from @smoothies.public.my_uploaded_files/fruits_available_for_smoothies.txt)
FILE_FORMAT = (FORMAT_NAME = smoothies.public.two_headerrow_pct_delim)
ON_ERROR = ABORT_STATEMENT
PURGE = TRUE;

In [ ]:
%%sql -r dataframe_6
use database SMOOTHIES;

create table orders (
ingredients varchar(200)
)

In [ ]:
%%sql -r dataframe_7
insert into smoothies.public.orders(ingredients) values ('Blueberries Elderberries ')

In [ ]:
%%sql -r dataframe_8
select * from smoothies.public.orders

In [ ]:
%%sql -r dataframe_9
truncate table smoothies.public.orders;

## Lesson 5 Prototype & Sprints

In [ ]:
%%sql -r dataframe_10
alter table smoothies.public.orders 
add column name_on_order varchar(100);


In [ ]:
%%sql -r dataframe_11
insert into smoothies.public.orders values ('Apples Cantaloupe ','Manu')

In [ ]:
%%sql -r dataframe_12
alter table smoothies.public.orders 
add column order_filled boolean default false;

update smoothies.public.orders
set order_filled = true
where name_on_order is null;

In [ ]:
%%sql -r dataframe_13
select * from smoothies.public.orders

## Lesson 6 : Pending Orders App Improvements 

In [ ]:
%%sql -r dataframe_14
use role sysadmin;
use database smoothies;
use warehouse COMPUTE_WH;

create sequence order_seq
    start = 1
    increment = 2
    comment = 'Provide a unique id for each smoothie order';

In [ ]:
%%sql -r dataframe_15
Truncate table smoothies.public.orders;

In [ ]:
%%sql -r dataframe_16
alter table SMOOTHIES.PUBLIC.ORDERS 
add column order_uid integer --adds the column
default smoothies.public.order_seq.nextval  --sets the value of the column to sequence
constraint order_uid unique enforced; --makes sure there is always a unique value in the column

In [ ]:
%%sql -r dataframe_17
create or replace table smoothies.public.orders (
       order_uid integer default smoothies.public.order_seq.nextval,
       order_filled boolean default false,
       name_on_order varchar(100),
       ingredients varchar(200),
       constraint order_uid unique (order_uid),
       order_ts timestamp_ltz default current_timestamp()
);

In [ ]:
alter table smoothies.public.fruit_options
add column search_on varchar(200) default null;

In [ ]:
%%sql -r dataframe_19
-- Apples, Blueberries, Jack Fruit, Raspberries and Strawberries.

update smoothies.public.fruit_options
set search_on = null
